In [0]:
pip install pls_common_data_store

In [0]:
dbutils.library.restartPython()

In [0]:
import os 
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp
import datetime as dt
from datetime import timedelta, datetime, date
from dateutil.relativedelta import relativedelta
import time 
from kayday import KrogerDate, DateRange
from effodata import ACDS, golden_rules, Joiner, Equality
import seg
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, Rollup, Cube, available_metrics, get_metrics
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
from pls_common_data_store import pls_data_store
import pyspark.sql.functions as F
from typing import List, Dict, Any

spark = SparkSession.builder.getOrCreate()

#### Training HH Universe
- Capture aggregated HH data from Nov 1, 2024 to Oct 1, 2025 as a training set to predict acquisition (Y) labels for 2025 Holiday HHs.
- Using acds history from previous holiday season + other key seasonal moments in KPF + more developed features to predict acquisition HHs in 25.
- Test set / Production set will be using Nov 1, 2025 - Aug, 2025 to identify lookalike HHs for live 2026 Holiday campaign.
- NOTE that I am able to currently extract Fall and Halloween related seasonal features for 2024-2025 model training, but 2025-2026 will come at a later date once the data is made available, since HH selection for Holiday is not until end of October.
- Use historical baseball segment KPF table to identify HHs who were gifting HHs or soon to be gifting HHs (does not appear the prev. month but appears next month), and exclude them from the training data, since we are targeting non-gifting HHs.
- This notebook is feature augmentation and engineering only. To decide what features to put into the model and the diagnostics/reasoning behind it, refer to eda notebook.

In [0]:
# Exclude HH from Current segment for real run, exclude HH from HISTORICAL segement for model training
gift_hh_segment_df_old = spark.read.parquet(f"abfss://kpf@sa8451cpkpfprd.dfs.core.windows.net/segments/current/tpg/gift_card_segment_HISTORICAL")

gift_hh_segment_df = spark.read.parquet(f"abfss://kpf@sa8451cpkpfprd.dfs.core.windows.net/segments/current/tpg/gift_card_segment_CURRENT")

##### To-Do: Derive which HHS converted to Gift HHs in 2025 Holiday and add training labels (0/1)
- The current training data scope is from Nov 2024 to Aug 2025.
- If HH spends on gift cards in November 2025, they become a gift HH in Dec 2025, and if a HH spends in December 2025, they become a gift HH in Jan 2026 (in the gift hh segment dataframe)
- Figure out which HHs were new HHs during this timeframe in 2025. 

In [0]:
# Gift HHs to exclude during training window

# Excluded HHs: all gift HHs in 2025 October and before
HHs_to_exclude_training = (
    gift_hh_segment_df_old.filter(f.col("fiscal_week") <= "20251004").select("ehhn", "fiscal_week", "gcs_seg_desc").distinct()
)

# Class Label (1) of Gift Buyers

# Holiday HHs: all gift HHs in November + December
holiday_2025_buyers = (
    gift_hh_segment_df_old
    .filter((f.col("fiscal_week") > "20251004") & (f.col("fiscal_week") <= "20251204")).select("ehhn", "fiscal_week", "gcs_seg_desc").distinct()
)

holiday_2025_buyers = holiday_2025_buyers.withColumn("target_label", f.lit(1))
holiday_2025_buyers.display()

# Get count of holiday_2025_buyers
holiday_2025_buyers.count()

In [0]:
# KPI session, DONT use sample mart for training data
use_sample_mart = False
acds = ACDS(use_sample_mart = use_sample_mart)
kpi_session = KPI(use_sample_mart = use_sample_mart)
pls_data_store_session = pls_data_store(spark)

In [0]:
dbutils.widgets.text("Training Start Date", "2024-11-01")
dbutils.widgets.text("Training Months Delta", "11")
run_date = dbutils.widgets.get("Training Start Date")
months_delta = int(dbutils.widgets.get("Training Months Delta"))
start_date = KrogerDate(date = run_date)

# change this delta from 9 months to whatever is needed based on current time (currently using 12 to account for Fall (Until end of October) in addition to Fathers Day, Summer, ect)
# month_delta = 11
new_date_str = (start_date.date + relativedelta(months=months_delta)).strftime('%Y-%m-%d')
end_date = KrogerDate(date=new_date_str)

print(f"Start Date Period: {start_date}")
print(f"End Date: {new_date_str}")
print(f"End Date Period: {end_date}")

In [0]:
# acds transaction data
acds_transactions = acds.get_transactions(
  start_date = start_date, end_date = end_date, join_with = ["products", "households"],
  apply_golden_rules = golden_rules(["store_exclusions", "customer_exclusions"])
)

# Remove all gift HHs up till 2025 November and December Gift HHs (Holiday) from training data
acds_transactions_no_gift_hhs = acds_transactions.join(HHs_to_exclude_training, on="ehhn",how="left_anti")

acds_transactions_no_gift_hhs.limit(5).display()

#### Features: ACDS Transactions by KPF Gifting Seasons (+ Campaign Live Dates)
- Holiday, Mothers Day / Spring / Easter, Fathers Day / Graduation / Summer, Valentines
- Only doing Q1 and Q2 gifting seasons for now, as targeting needs to be done in September, and time scope in 2024-2025 and 2025-2026 train/test need to be symmetrical.

In [0]:
# Feature Creation: Net Spend, Total Trips, Division, and Spend + Trips + Share Percentage Per Seasonal Window (Holiday, Valentines, Mothers Day/Spring/Easter, Fathers Day/Grad/Summer)
def create_seasonal_household_features(
    df,
    windows: List[Dict[str, str]],
    id_col: str = "ehhn",
    date_col: str = "trn_dt",
    spend_col: str = "net_spend_amt",
    div_col: str = "mgt_div_no" 
):
    # total trips, sum of net spend, and most frequent division
    agg_expressions = [
        f.sum(spend_col).alias("total_net_spend"),
        f.countDistinct(date_col).alias("total_trips"),
        f.mode(div_col).alias("primary_mgt_div_no") # Calculates the mode (most frequent value)
    ]

    # Aggregations per Seasonal Window
    for window in windows:
        prefix = window["prefix"]
        start = window["start"]
        end = window["end"]
    
    # Net Spend per Seasonal Window
        agg_expressions.append(
            f.sum(
                f.when(
                    (f.col(date_col) >= start) & (f.col(date_col) <= end),
                    f.col(spend_col),
                ).otherwise(0)
            ).alias(f"{prefix}_net_spend")
        )

    # Trip Count per Seasonal Window
        agg_expressions.append(
            f.countDistinct(
                f.when(
                    (f.col(date_col) >= start) & (f.col(date_col) <= end),
                    f.col(date_col),
                )
            ).alias(f"{prefix}_trips")
        )
        
    feature_df = df.groupBy(id_col).agg(*agg_expressions)

    # Share Percentage Per Season (What % of their spendings were done in Holiday, for example)
    for w in windows:
        prefix = w["prefix"]
        feature_df = feature_df.withColumn(
            f"{prefix}_spend_pct",
            f.when(
                f.col("total_net_spend") > 0,
                (f.col(f"{prefix}_net_spend") / f.col("total_net_spend")) * 100,
            ).otherwise(0),
        )

    return feature_df

In [0]:
# Method Call
seasonal_windows = [
    {
        "prefix": "holiday_2024_kpf", 
        "start": "20241112", 
        "end": "20241231"
    },

    {
        "prefix": "spr_eas_mothers_2025_kpf", 
        "start": "20250320", 
        "end": "20250520"
    },
        
    {
        "prefix": "fathers_summer_grad_2025_kpf", 
        "start": "20250520", 
        "end": "20250710"
    },

    {
        "prefix": "vtines_2025_kpf", 
        "start": "20250201", 
        "end": "20250220"
    },

    {
        "prefix": "fall_2025_kpf", 
        "start": "20250920", 
        "end": "20251020"
    },
]

# Run feature generation
hh_feature_matrix = create_seasonal_household_features(
    df=acds_transactions_no_gift_hhs,
    windows=seasonal_windows,
    id_col="ehhn",
    date_col="trn_dt",
    spend_col="net_spend_amt",
    div_col= "mgt_div_no" 
)

# Set any pct column < 0 to 0
pct_cols = [c for c in hh_feature_matrix.columns if "pct" in c.lower()]
for col in pct_cols:
    hh_feature_matrix = hh_feature_matrix.withColumn(
        col, f.when(f.col(col) < 0, 0).otherwise(f.col(col))
    )

hh_feature_matrix.display()

In [0]:
# Quick conversion of division to categorical
hh_feature_matrix = hh_feature_matrix.withColumn("primary_mgt_div_no", f.col("primary_mgt_div_no").cast("string"))

In [0]:
hh_feature_matrix.count()

#### Features: Fuel Point Activity
- We will use points earned, as points redeemed and burn rate will change drastically in Q3 due to the launch of Kroger Credit Card
- The logic is that HHs who earn lots of fuel points (and particularly during key gifting seasons) but are not gift HHs might be good lookalike HHs, and KPF campaigns can be used to further reinforce certain habits (such as HHs who like or tend to earn lots of fuel points in Holiday, for example)

In [0]:
# fuel points table
pls_data_store_session = pls_data_store(spark)
fuel_table = pls_data_store_session.get_points_detail(
  start_date=run_date, end_date=new_date_str, query_filters=["type == 'FUEL'"])
fuel_table = fuel_table.filter(f.col("points_earned") > 0)
fuel_table.display()

In [0]:
# Feature Creation: Fuel Points earned, and Fuel Points Earned + Share Percentage Per Seasonal Window (Holiday, Valentines, Mothers Day/Spring/Easter, Fathers Day/Grad/Summer)
# Remove all Gifting HHs from fuel table. These HHs are non-gift buyers and their fuel point activity -> this could give us good insights for lookalike

def create_seasonal_fuel_features(
    df,
    windows: List[Dict[str, str]],
    id_col: str = "ehhn",
    date_col: str = "trn_dt",
    points_col: str = "points_earned"
):

    # Total Fuel Points earned
    agg_expressions = [
        f.sum(points_col).alias("total_fuel_points")
    ]
    
    # Points earned during each seasonal window
    for w in windows:
        prefix = w["prefix"]
        start = w["start"]
        end = w["end"]
        
        agg_expressions.append(
            f.sum(
                f.when(
                    (f.col(date_col) >= start) & (f.col(date_col) <= end),
                    f.col(points_col)
                ).otherwise(0)
            ).alias(f"{prefix}_fuel_points")
        )
    feature_df = df.groupBy(id_col).agg(*agg_expressions)
    
    # Share percentage of fuel points earned per seasonal window
    pct_expressions = [
        f.when(
            f.col("total_fuel_points") > 0,
            (f.col(f"{w['prefix']}_fuel_points") / f.col("total_fuel_points")) * 100
        ).otherwise(0).alias(f"{w['prefix']}_fuel_points_pct")
        for w in windows
    ]
    
    return feature_df.select("*", *pct_expressions)

In [0]:
seasonal_windows = [
    {
        "prefix": "holiday_2024_kpf",
        "start": "20241112",
        "end": "20241231"
    },

    {
        "prefix": "spr_eas_mothers_2025_kpf",
        "start": "20250320",
        "end": "20250520"
    },

    {
        "prefix": "fathers_summer_grad_2025_kpf",
        "start": "20250520",
        "end": "20250710"
    },

    {
        "prefix": "vtines_2025_kpf",
        "start": "20250201",
        "end": "20250220"
    },

    {
        "prefix": "fall_2025_kpf", 
        "start": "20250920", 
        "end": "20251020"
    },
]

# Run the fuel feature generation method
fuel_features_df = create_seasonal_fuel_features(
    df=fuel_table,
    windows=seasonal_windows,
    id_col="ehhn",
    date_col="trn_dt",
    points_col="points_earned"
)

fuel_features_df.display()

In [0]:
# Fuel points table here will auto-exclude all gift HHs, since we are left joining to acds data that was previously anti-joined with historical gift hhs 
fuel_cols_to_impute = [c for c in fuel_features_df.columns if c != "ehhn"]
lookalike_features = (
    hh_feature_matrix
    .join(fuel_features_df, on="ehhn", how="left")
    .filter(f.col("total_net_spend") > 0)
    .fillna(0, subset=fuel_cols_to_impute)
)

lookalike_features.display()

#### Features: Greeting Card Transactions + Trips + Days Between Trips
- Roughly 70% of gift card buyers were greeting card buyers in 2025 (but only 40% vise versa). Avenue to tap into greeting card buyers
who are not gift buyers and convert them could be possible.

In [0]:
# Greeting card transactions in acds
acds_transactions_greeting_cards = acds.get_transactions(
  start_date = run_date, end_date = new_date_str, join_with = ["products", "households"],
  apply_golden_rules=golden_rules(["store_exclusions", "customer_exclusions"]),
  query_filters= ["pid_fyt_com_cd IN ('235')",
        "mgt_div_no in ('011','014','016','018','021','024','025','026','029','034','035','531','534','615','620','660','701','703','705','706')",
        "ehhn is not null",
        "net_spend_amt >= 0",
        "scn_unt_qy >= 0"]
)
acds_transactions_greeting_cards.display()

In [0]:
# Avg days between greeting card transactions for EHHNs (can be a good feature, repeat HHs)
# if a HH only went once, made a flag to represent that (maybe not the type of HH we should be looking-alike)

acds_trips_distinct = (
    acds_transactions_greeting_cards
    .groupBy("ehhn", "trn_dt")
    .agg(f.sum("net_spend_amt").alias("trip_net_spend"))
)
window_spec = Window.partitionBy("ehhn").orderBy("trn_dt")

acds_trips_with_lag = (
    acds_trips_distinct
    .withColumn("prev_trn_dt", f.lag("trn_dt").over(window_spec))
    .withColumn(
        "diff_days", 
        f.datediff(
            f.to_date(f.col("trn_dt"), "yyyyMMdd"), 
            f.to_date(f.col("prev_trn_dt"), "yyyyMMdd")
        )
    )
)

# Features: Total spent on greeting card, avg spent per trip, avg days between trips, total trips involving a greeting card, and whether a HH is a one time greeting card buyer (repeat buyers might fit lookalike profile better)
avg_diff_df = (
    acds_trips_with_lag
    .groupBy("ehhn")
    .agg(
        f.sum("trip_net_spend").alias("total_greeting_card_spend"),
        f.avg("trip_net_spend").alias("avg_greeting_card_spend_per_trip"),
        f.avg("diff_days").alias("avg_days_between_trips"),
        f.count("trn_dt").alias("greeting_card_trips_count"),
        f.when(f.count("diff_days") == 0, True).otherwise(False).alias("is_one_time_greeting_card_buyer")
    )
)

# Coalesce 0s in avg days between trips. Usually if a HH is a one time greeting card buyer, it will be 0 -> the flag will tell us 
greeting_card_features = (
    avg_diff_df
    .withColumn(
        "avg_days_between_greeting_card_buys",
        f.coalesce(f.col("avg_days_between_trips"), f.lit(0))
    )
    .select(
        "ehhn",
        "total_greeting_card_spend",
        "avg_greeting_card_spend_per_trip",
        "avg_days_between_greeting_card_buys",
        "greeting_card_trips_count",
    )
)

display(greeting_card_features)

In [0]:
# Joining greeting card features with Fuel and ACDS Seasonality Features
greeting_cols_to_impute = [c for c in greeting_card_features.columns if c != "ehhn"]
lookalike_features_v2 = (lookalike_features
    .join(greeting_card_features, on="ehhn", how="left")
    .fillna(0, subset=greeting_cols_to_impute)
)

lookalike_features_v2.display()

#### Features: Customer Dimensions + Digital Engagement Segmentations
- Wondering if customer dimensions can tell us something about acquisition HHs, since there is a trend of top performing card partners in the TDC program, which may tie into consumer habits (usually Lowes + Fast Food)
- HHs who are highly engaged with Kroger digital platforms. Worth investigating whether these HHs are more receptive to various KPF tactics (since most KPF campaigns are digital, such as SSE, TDC, MCP SSE, DISP, ect)

In [0]:
# customer dimensions on date
cds = seg.get_seg_for_date('cds_4_hh', new_date_str)
cds_relevant_dim_seg = cds.select(
  "ehhn", "price_dim_seg", "health_dim_seg", "convenience_dim_seg", "variety_seeking_dim_seg")

lookalike_features_v3 = lookalike_features_v2.join(cds_relevant_dim_seg, on="ehhn", how="left")
lookalike_features_v3.display()

In [0]:
total_rows = lookalike_features_v3.count()
null_price_dim_seg_rows = lookalike_features_v3.filter(f.col("price_dim_seg").isNull()).count()

print(f"Total rows: {total_rows}")
print(f"Rows with null price_dim_seg: {null_price_dim_seg_rows}")

In [0]:
dig_eng = seg.get_seg_for_date('digital_engagement', new_date_str).select("ehhn", "dig_eng_seg_code")

lookalike_features_v4 = lookalike_features_v3.join(dig_eng, on="ehhn", how="left")
lookalike_features_v4.display()

In [0]:
total_rows = lookalike_features_v4.count()
null_dig_seg_rows = lookalike_features_v4.filter(f.col("dig_eng_seg_code").isNull()).count()

print(f"Total rows: {total_rows}")
print(f"Rows with null price_dim_seg: {null_dig_seg_rows}")

In [0]:
# Inputing null elements so h2o doesnt crash
seg_cols = [c for c in lookalike_features_v4.columns if "seg" in c.lower()]
lookalike_features_v4 = lookalike_features_v4.fillna("Unassigned", subset=seg_cols)

# ~37 Features
display(lookalike_features_v4)

#### Target Labels for Nov/Dec Gift HHs vs. Non Gift HHs
- Removed all Gifting HHs prior to Holiday Season. This means the only Hhs left are non gifting HHs, or HHs who bought a gift card in Nov / Dec.
- Joining holiday 2025 baseball segments to this df which contains target label for classification.

In [0]:
lookalike_features_vfinal = (
    lookalike_features_v4
    .join(holiday_2025_buyers.select("ehhn", "target_label"), on="ehhn", how="left")
    .fillna({"target_label": 0})
)

lookalike_features_vfinal.display()
lookalike_features_vfinal.groupBy("target_label").count().display()

In [0]:
# Training and Test Data Stratified Split based on target labels
# Target Label is heavily skewed towards 0, since there are a lot more people who are not gift buyers in Holiday vs. people who converted to gift buyers in nov/dec
train_fraction = 0.8
fractions = {0: train_fraction, 1: train_fraction}

# Create Stratified Training Set (80% of 0s and 80% of 1s)
train_df = lookalike_features_vfinal.sampleBy("target_label", fractions=fractions, seed=42)

test_df = lookalike_features_vfinal.join(
    train_df.select("ehhn"), on="ehhn", how="left_anti"
)

train_df.groupBy("target_label").agg(
    f.count("*").alias("hh_count"),
    f.round((f.count("*") / train_df.count()) * 100, 2).alias("percentage")
).display()

test_df.groupBy("target_label").agg(
    f.count("*").alias("hh_count"),
    f.round((f.count("*") / test_df.count()) * 100, 2).alias("percentage")
).display()

In [0]:
# Write to data shuttle and output for testing
train_df.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_trainingdata')

test_df.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_testdata')
